# Compare sleep phenotypes between groups of different diseases

In [ ]:
import numpy as np
import pandas as pd

import seaborn as sns
import pylab as plt

from scipy import stats
from statannot import add_stat_annotation
import joblib
#import pingouin as pg

from importlib import reload
import sys
import os
sys.path.insert(1,'../')
import plots

In [ ]:
reload(plots)
image_path = plots.set_dir()
colors = plots.set_colors()
data_path = '/scratch/c.c21013066/data/ukbiobank/phenotypes/accelerometer/'

In [ ]:
features_res = ['mean_sleep_hours_per24h_residual','std_movement_during_sleep_residual','mean_max_sleep_hours_consecutive_per24h_residual',
            'mean_N_sleep_intervals_22-10_residual','mean_N_sleep_intervals_10-22_residual','mean_N_sleep_intervals_23-07_residual','mean_N_sleep_intervals_07-23_residual']
rename_dict = {'mean_sleep_hours_per24h_residual':'mean sleep [h]','std_movement_during_sleep_residual':'std movement during sleep [milligal]',
                        'mean_max_sleep_hours_consecutive_per24h_residual':'mean maximum consecutive sleep [h]',
            'mean_N_sleep_intervals_22-10_residual':'mean wake-ups 10pm-10am [N]',
                           'mean_N_sleep_intervals_10-22_residual':'mean naps 10am-10pm [N]',
                          'mean_N_sleep_intervals_23-07_residual':'mean wake-ups 11pm-07am [N]',
                           'mean_N_sleep_intervals_07-23_residual':'mean naps 07am-11pm [N]'}
covs = ['diagnosis','Status','eid']
diseases = ['ParkinsonDisease']


dfs_res = pd.read_csv(f'{data_path}unmatched_all_sleepresidual_BMI_HCnoOsteo_intercept.csv',index_col=0)
noDepnoPD = pd.read_csv(f'{data_path}unmatched_all_NoDepressedNoPD_sleepresidual_BMI_HCnoOsteo_intercept.csv',index_col=0)

data_res = dfs_res.reset_index()[np.hstack([features_res,covs])]
data_res= data_res.rename(columns=rename_dict)
noDepnoPD = noDepnoPD.reset_index()[np.hstack([features_res,covs])]
noDepnoPD= noDepnoPD.rename(columns=rename_dict)

datas = [data_res,noDepnoPD]

hc_bmi = pd.read_csv(f'{data_path}//HCnoOsteo_sleepresiduals_bmi_intercept.csv',
                      index_col=0)
hc_bmi = hc_bmi.reset_index()[features_res]
hc_bmi = hc_bmi.rename(columns=rename_dict)

features_res = ['mean sleep [h]','std movement during sleep [milligal]','mean maximum consecutive sleep [h]',
            'mean wake-ups 10pm-10am [N]','mean naps 10am-10pm [N]','mean wake-ups 11pm-07am [N]','mean naps 07am-11pm [N]']
features_res = ['mean sleep [h]','std movement during sleep [milligal]','mean maximum consecutive sleep [h]',
                'mean wake-ups 11pm-07am [N]','mean naps 07am-11pm [N]']
feature_labels = ['mean sleep [h]','std movement during \nsleep [milligal]','mean maximum \nconsecutive sleep [h]',
                'mean wake-ups \n11pm-07am [N]','mean naps \n07am-11pm [N]']

In [ ]:
names = ['all_res','NoDepressedNoPD']
yss = [features_res,features_res]
subplotlabels = ['A','B','C','D','E']#,"F",'G']

for name,df,ys in zip(names,datas,yss):
    fig,axes = plt.subplots(figsize=(15,25),nrows=5,sharex=True)
    plots.plot_context()
    for y,label,ax,ylabel in zip(ys,subplotlabels,axes.flatten(),feature_labels):

            subset = ['AllCauseDementia','AlzheimerDisease','AllCauseParkinsonism','ParkinsonDisease','Dystonia','Osteoarthritis','Depression']
            data = df[df['diagnosis'].isin(subset)]
            data['Status'] = data['Status'].replace(['Diseased'],['Diagnosed'])
            data = data[data['Status']!='Healthy']
            # get all HC
            hc = hc_bmi.dropna(subset=[y],axis='rows',how='any')
            hc['Status'] = 'Healthy'
            # only use unique set of matched HC
            #hc = data[data['Status']=='Healthy']
            #hc = hc[~hc.index.duplicated(keep='first')]
            data=data[data['Status']!='Healthy']
            for diagnosis in subset:
                hc['diagnosis'] = diagnosis
                data = pd.concat([data,hc])

            thresh = 0.05/(len(subset)*3)
            annot = [[2e-6, "****"], [2e-5, "***"], [2e-4, "**"], [thresh, "*"], [1, "ns"]]
            box_pairs = []
            statistics = pd.DataFrame(columns=pd.MultiIndex.from_product([['T','dof','alternative','p-value','CI95%',
                                                           'cohen-d','BF10','power'],subset],
                                                            names = ['statistic','disorder']),
                                              index=pd.MultiIndex.from_product([['Healthy','Prodromal','Diagnosed'],
                                                          ['Healthy','Prodromal','Diagnosed']],
                                                         names = ['g1','g2']))
            ttests = pd.DataFrame(index=pd.MultiIndex.from_product([['Healthy','Prodromal','Diagnosed'],
                                                                       ['Healthy','Prodromal','Diagnosed']],
                                                                      names = ['g1','g2']),columns=['t','p'])
            for diagnosis in subset:
                diag = data[data['diagnosis']==diagnosis]
                ttest = pd.DataFrame(index=pd.MultiIndex.from_product([['Healthy','Prodromal','Diagnosed'],
                                                                       ['Healthy','Prodromal','Diagnosed']],
                                                                      names = ['g1','g2']),columns=['t','p'])
                for i,g1 in enumerate(['Healthy','Prodromal','Diagnosed']):
                    for j,g2 in enumerate(['Healthy','Prodromal','Diagnosed']):
                        if j>i:
                            try:
                                statistics.loc[(g1,g2),(slice(None),diagnosis)] = pg.ttest(diag.loc[diag['Status']==g1,y],
                                                                                          diag.loc[diag['Status']==g2,y],correction=np.logical_or(g1=='Healthy',g2=='Healthy')).values
                                t,p = stats.ttest_ind(diag.loc[diag['Status']==g1,y],diag.loc[diag['Status']==g2,y],equal_var=np.logical_or(g1=='Healthy',g2=='Healthy'))
                                ttest.loc[(g1,g2),'t'] = t#statistics.loc[(g1,g2),('T',diagnosis)][0]
                                ttest.loc[(g1,g2),'p'] = p#statistics.loc[(g1,g2),('p-value',diagnosis)][0]
                            except:
                                print(diagnosis,g1,g2)
                ttest = ttest.dropna(axis='rows',how='all')
                sign = ttest[ttest['p']<thresh]
                for key,row in sign.iterrows():
                    box_pairs.append(((diagnosis,key[0]),(diagnosis,key[1])))
            
            N_diagnosis = data.dropna(subset=[y]).groupby(['diagnosis','Status']).size()
            N_diagnosis = N_diagnosis.loc[(['AllCauseDementia','AlzheimerDisease','AllCauseParkinsonism','ParkinsonDisease','Dystonia',
                                           'Osteoarthritis','Depression'],['Healthy','Prodromal','Diagnosed'])]

            bp = sns.boxplot(y=y,hue='Status',x='diagnosis',data=data,hue_order=['Healthy','Prodromal','Diagnosed'],showfliers=False,
                            order=subset,ax=ax)
            if len(box_pairs)>0:
                ax, test_results = add_stat_annotation(ax, data=data, x='diagnosis', y=y, hue='Status',hue_order=['Healthy','Prodromal','Diagnosed'],
                                                   order=subset,
                                               box_pairs=box_pairs,
                                               test='t-test_ind', text_format='star', loc='outside', verbose=2,comparisons_correction=None,
                                                  pvalue_thresholds=annot)
            statistics = statistics.dropna(axis='rows',how='all')
            statistics.to_csv(f'/scratch/c.c21013066/data/ukbiobank/phenotypes/accelerometer/group_differences_{y}_{name}_HCnoOsteo_intercept.csv')
            ax.set_ylabel(f'{ylabel} residual')
            ax.set_xlabel('')
            if label==subplotlabels[-1]:
                ax.set_xticklabels(['AllCauseDementia','AlzheimerDisease','OtherParkinsonism',
                                    'ParkinsonDisease','Dystonia','Osteoarthritis','Depression'],rotation=90)
            else:
                ax.tick_params(bottom = False)
            if label=='A':
                ax.legend(bbox_to_anchor=(1.01, 1), loc=2, borderaxespad=0.);
            else:
                ax.legend([],[], frameon=False)
            remove = len(box_pairs)
            plots.add_median_labels(ax,N_diagnosis,fmt="%d",remove=remove)
            ax.text(-0.1, 1.1,label, transform=ax.transAxes, 
            size=20, weight='bold')
    plt.tight_layout()
    plt.savefig(f'{image_path}/diagsMatch_{y}_boxplot_{name}resBMI_allHCnoOsteo_intercept.pdf',bbox_inches='tight',dpi=300)
    plt.savefig(f'{image_path}/diagsMatch_{y}_boxplot_{name}resBMI_allHCnoOsteo_intercept.png',bbox_inches='tight',dpi=300)
    plt.show()